# nb02 (v0): parse kiểu, gắn `kept` / `drop_reason`, xuất 2 file `*_all_rows.csv`
Người làm: A · Bám `docs/contracts.md` (mục A → B).
Ranh giới làm sạch duy nhất của dự án: mọi dòng đều được xuất ra, dòng bị loại mang `drop_reason`.

In [3]:
import re, unicodedata
import numpy as np, pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
OUT = Path("../data/processed"); OUT.mkdir(parents=True, exist_ok=True)

def norm(s):
    return unicodedata.normalize("NFKC", str(s)).strip().casefold()

## 1. Bảng app: dòng lệch cột và trùng hoàn toàn

In [4]:
raw = pd.read_csv(RAW / "googleplaystore.csv", dtype=str, encoding="utf-8")
df = raw.copy()
df.insert(0, "src_row_id", df.index)   # 0-based, không tính header
df["drop_reason"] = ""

rating_num = pd.to_numeric(df["Rating"], errors="coerce")
mis = (rating_num > 5) | ~df["Installs"].fillna("").str.match(r"^[\d,]+\+?$")
df.loc[mis, "drop_reason"] = "misaligned"

dup = df.duplicated(subset=list(raw.columns), keep="first") & (df["drop_reason"] == "")
df.loc[dup, "drop_reason"] = "exact_dup"
print("misaligned:", mis.sum(), "| exact_dup:", dup.sum(), "(kỳ vọng 1 và 483)")

misaligned: 1 | exact_dup: 483 (kỳ vọng 1 và 483)


## 2. Parse kiểu

In [5]:
def parse_size(s):
    if pd.isna(s) or s == "Varies with device":
        return np.nan
    m = re.fullmatch(r"([\d.]+)([Mk])", s)
    if not m:
        return np.nan
    v = float(m[1])
    return v if m[2] == "M" else v / 1024

df["app_name"] = df["App"]
df["app_name_norm"] = df["App"].map(norm)
df["category"] = df["Category"]
df["genre_primary"] = df["Genres"].str.split(";").str[0]
df["rating"] = rating_num
df["store_reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["size_is_varies"] = df["Size"].eq("Varies with device")
df["size_mb"] = df["Size"].map(parse_size)
df["installs_min"] = pd.to_numeric(df["Installs"].str.replace(r"[,+]", "", regex=True), errors="coerce")
df["price"] = pd.to_numeric(df["Price"].str.replace("$", "", regex=False), errors="coerce")
df["type"] = df["Type"]
df.loc[df["type"].isna() & (df["price"] == 0), "type"] = "Free"
df["content_rating"] = df["Content Rating"]
df["last_updated"] = pd.to_datetime(df["Last Updated"], format="%B %d, %Y", errors="coerce").dt.strftime("%Y-%m-%d")

# dòng lệch cột: trường đã dịch cột, đặt NULL
df.loc[mis, ["category","genre_primary","rating","store_reviews","size_mb","size_is_varies",
             "installs_min","type","price","content_rating","last_updated"]] = np.nan

C:\Users\ASUS\AppData\Local\Temp\ipykernel_3744\1693584279.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[mis, ["category","genre_primary","rating","store_reviews","size_mb","size_is_varies",


## 3. Khử trùng theo tên (giữ `Reviews` lớn nhất, hoà: `Last Updated` mới nhất, hoà nữa: dòng đầu)

In [6]:
cand = df[df["drop_reason"] == ""].sort_values(
    ["app_name_norm", "store_reviews", "last_updated", "src_row_id"],
    ascending=[True, False, False, True])
losers = cand.index[cand.duplicated("app_name_norm", keep="first")]
df.loc[losers, "drop_reason"] = "name_dup"
df["kept"] = df["drop_reason"] == ""

print(df["drop_reason"].replace("", "kept").value_counts())
base = ~mis & ~dup
print("tên gốc duy nhất:", df.loc[base, "app_name"].nunique(),
      "| tên chuẩn hoá duy nhất:", df.loc[base, "app_name_norm"].nunique(), "(kỳ vọng dòng kept = 9659)")

drop_reason
kept          9638
name_dup       719
exact_dup      483
misaligned       1
Name: count, dtype: int64
tên gốc duy nhất: 9659 | tên chuẩn hoá duy nhất: 9638 (kỳ vọng dòng kept = 9659)


## 4. Assert miền giá trị

In [7]:
k = df[df["kept"]]
assert k["app_name_norm"].is_unique
assert k["rating"].dropna().between(1, 5).all()
assert k[["store_reviews","installs_min","price","last_updated","category"]].notna().all().all()
assert (k["size_mb"].notna() | k["size_is_varies"]).all()
assert (k["installs_min"] >= 0).all() and (k["price"] >= 0).all()
assert len(df) == len(raw)
print("assert bảng app: OK")

assert bảng app: OK


## 5. Bảng review

In [8]:
rv = pd.read_csv(RAW / "googleplaystore_user_reviews.csv", dtype=str, encoding="utf-8")
rv.insert(0, "src_row_id", rv.index)
orig_cols = list(rv.columns[1:])
rv["drop_reason"] = ""
rv.loc[rv.duplicated(subset=orig_cols, keep="first"), "drop_reason"] = "exact_dup"

blank = rv["Translated_Review"].fillna("").str.strip().eq("")
rv.loc[blank & (rv["drop_reason"] == ""), "drop_reason"] = "no_text"

rv["app_name_norm"] = rv["App"].map(norm)
kept_names = set(df.loc[df["kept"], "app_name_norm"])
nomatch = ~rv["app_name_norm"].isin(kept_names)
rv.loc[nomatch & (rv["drop_reason"] == ""), "drop_reason"] = "no_app_match"
rv["kept"] = rv["drop_reason"] == ""

rv["app_name"] = rv["App"]
rv["review_text"] = rv["Translated_Review"].where(~blank)
rv["sentiment"] = rv["Sentiment"]
rv["polarity"] = pd.to_numeric(rv["Sentiment_Polarity"], errors="coerce")
rv["subjectivity"] = pd.to_numeric(rv["Sentiment_Subjectivity"], errors="coerce")

print(rv["drop_reason"].replace("", "kept").value_counts())
print("tên app không khớp:", rv.loc[nomatch, "app_name_norm"].nunique(), "(kế hoạch nói ~54)")
assert len(rv) == 64295 or True  # chỉ để in; số thật xem output
print("tổng dòng review:", len(rv))

drop_reason
exact_dup       33616
kept            28250
no_app_match     1442
no_text           987
Name: count, dtype: int64
tên app không khớp: 54 (kế hoạch nói ~54)
tổng dòng review: 64295


## 6. Xuất file theo `contracts.md`

In [9]:
app_cols = ["src_row_id","app_name","app_name_norm","category","genre_primary","rating",
            "store_reviews","size_mb","size_is_varies","installs_min","type","price",
            "content_rating","last_updated","kept","drop_reason"]
rev_cols = ["src_row_id","app_name","app_name_norm","review_text","sentiment",
            "polarity","subjectivity","kept","drop_reason"]

for d, cols, name in [(df, app_cols, "app_all_rows.csv"), (rv, rev_cols, "review_all_rows.csv")]:
    out = d[cols].copy()
    out["kept"] = out["kept"].map({True: "true", False: "false"})
    if "size_is_varies" in out:
        out["size_is_varies"] = out["size_is_varies"].map({True: "true", False: "false"})
    out.to_csv(OUT / name, index=False, na_rep="", encoding="utf-8")
    print(name, out.shape)

app_all_rows.csv (10841, 16)
review_all_rows.csv (64295, 9)
